In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile
import hashlib
import io
from PIL import Image

print("Ready")

Ready


In [2]:
from google.colab import files

uploaded = files.upload()

Saving DVM_Project3_Review_20261114_201843_029683.zip to DVM_Project3_Review_20261114_201843_029683.zip


In [3]:
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert len(zip_names) == 1, "Upload exactly one ZIP file."

source_zip = Path(zip_names[0])

with zipfile.ZipFile(source_zip) as archive:
    candidates = pd.read_csv(
        io.BytesIO(archive.read("selection_manifest.csv")),
        dtype=str
    ).fillna("")

print("Photos:", len(candidates))
print("Listings:", candidates["Adv_ID"].nunique())

display(
    candidates.groupby("Body_type").agg(
        Listings=("Adv_ID", "nunique"),
        Photos=("Image_ID", "size"),
        Models=("Genmodel_ID", "nunique")
    )
)

Photos: 4500
Listings: 1500


,Listings,Photos,Models
Body_type,,,
Pickup,500,1500,13
SUV,500,1500,144
Sedan,500,1500,118


In [4]:
review_upload = files.upload()

review_zips = [
    name for name in review_upload
    if name.lower().endswith(".zip")
]
assert len(review_zips) == 1, "Upload only DVM_Review_Records.zip."

with zipfile.ZipFile(review_zips[0]) as archive:
    review = pd.read_csv(
        io.BytesIO(archive.read("listing_review.csv")),
        dtype=str
    ).fillna("")

assert review["Adv_ID"].is_unique
assert set(review["Adv_ID"]) == set(candidates["Adv_ID"])

display(pd.crosstab(review["Body_type"], review["Review_decision"]))

Saving DVM_Review_Records.zip to DVM_Review_Records.zip


Review_decision,provisional_keep,reject,uncertain
Body_type,,,
Pickup,94,295,111
SUV,371,72,57
Sedan,395,47,58


In [5]:
keep_ids = set(
    review.loc[
        review["Review_decision"].eq("provisional_keep"),
        "Adv_ID"
    ]
)

clean = candidates[
    candidates["Adv_ID"].isin(keep_ids)
].copy()

clean["Review_ID"] = clean["Adv_ID"].map(
    review.set_index("Adv_ID")["Review_ID"]
)

view_map = {
    "0": "Front",
    "90": "Side",
    "270": "Side",
    "180": "Rear"
}

assert clean["Image_ID"].is_unique
assert clean["File_SHA256"].is_unique
assert clean["Predicted_viewpoint"].map(view_map).eq(clean["View"]).all()
assert not clean["Quality_check"].str.upper().eq("N").any()

assert clean["Image_ID"].str.rsplit(
    "$$", n=1
).str[0].eq(clean["Adv_ID"]).all()

assert clean.groupby("Adv_ID")["Genmodel_ID"].nunique().eq(1).all()
assert clean.groupby("Adv_ID")["Body_type"].nunique().eq(1).all()
assert clean.groupby("Adv_ID").size().eq(3).all()

assert clean.groupby("Adv_ID")["View"].agg(set).map(
    lambda views: views == {"Front", "Side", "Rear"}
).all()

assert len(clean) == 2580
assert clean["Adv_ID"].nunique() == 860

display(
    clean.groupby("Body_type").agg(
        Listings=("Adv_ID", "nunique"),
        Photos=("Image_ID", "size"),
        Models=("Genmodel_ID", "nunique")
    )
)

,Listings,Photos,Models
Body_type,,,
Pickup,94,282,9
SUV,371,1113,126
Sedan,395,1185,107


In [7]:
import unicodedata
import re
import json

output_dir = Path("/content/DVM_Cleaned_Data")
photo_dir = output_dir / "Photos"
photo_dir.mkdir(parents=True, exist_ok=True)

def safe_name(value):
    value = unicodedata.normalize(
        "NFKD", str(value)
    ).encode("ascii", "ignore").decode()

    return re.sub(
        r"[^A-Za-z0-9]+", "_", value
    ).strip("_")

view_order = {"Front": 1, "Side": 2, "Rear": 3}

clean["Review_ID"] = clean["Review_ID"].astype(int)
clean["_view_order"] = clean["View"].map(view_order)

clean = clean.sort_values(
    ["Review_ID", "_view_order"]
).reset_index(drop=True)

records = []
pixel_hashes = set()
expected_names = set()

with zipfile.ZipFile(source_zip) as archive:
    for _, row in clean.iterrows():
        raw = archive.read(row["Extracted_path"])

        assert hashlib.sha256(raw).hexdigest() == row["File_SHA256"]

        with Image.open(io.BytesIO(raw)) as image:
            image.load()
            assert image.size == (300, 300), row["Image_ID"]

            pixel_hash = hashlib.sha256(
                image.convert("RGB").tobytes()
            ).hexdigest()

        assert pixel_hash not in pixel_hashes, row["Image_ID"]
        pixel_hashes.add(pixel_hash)

        filename = (
            f'{row["Review_ID"]:04d}_'
            f'{safe_name(row["Body_type"])}_'
            f'{safe_name(row["Maker"])}_'
            f'{safe_name(row["Genmodel"])}_'
            f'{view_order[row["View"]]:02d}_'
            f'{row["View"]}_'
            f'{safe_name(row["Adv_ID"])}.jpg'
        )

        assert filename not in expected_names
        expected_names.add(filename)

        (photo_dir / filename).write_bytes(raw)

        columns = [
            "Review_ID", "Body_type", "Maker", "Genmodel",
            "Genmodel_ID", "Adv_ID", "View", "Image_ID",
            "Image_name", "Predicted_viewpoint",
            "Quality_check", "Source_relative_path"
        ]

        record = {column: row[column] for column in columns}

        record.update({
            "Extracted_path": "Photos/" + filename,
            "File_SHA256": row["File_SHA256"],
            "Class_id": {
                "Sedan": 0, "SUV": 1, "Pickup": 2
            }[row["Body_type"]],
            "Filename": filename,
            "Human_review_status": "approved",
            "Human_reviewer": "Shawn Chen",
            "Human_review_date": "2026-11-15"
        })

        records.append(record)

    (output_dir / "DATA_LICENSE.txt").write_bytes(
        archive.read("DATA_LICENSE.txt")
    )

assert {
    path.name for path in photo_dir.iterdir()
} == expected_names, "The output folder contains unexpected files."

clean = pd.DataFrame(records)

clean.to_csv(
    output_dir / "clean_manifest.csv",
    index=False
)

review.to_csv(
    output_dir / "listing_review.csv",
    index=False
)

print("Verified and saved:", len(clean), "photos")

Verified and saved: 2580 photos


In [10]:
seed = 4002
trials = 5000

classes = ["Sedan", "SUV", "Pickup"]
split_names = np.array(["train", "validation", "test"])
target = np.array([0.70, 0.15, 0.15])

listings = clean.drop_duplicates("Adv_ID")

model_counts = pd.crosstab(
    listings["Genmodel_ID"],
    listings["Body_type"]
).reindex(
    columns=classes,
    fill_value=0
).sort_index()

primary_body = model_counts.idxmax(axis=1).to_numpy()
counts = model_counts.to_numpy()
class_totals = counts.sum(axis=0)

def allocate_models(number):
    if number < 3:
        raise ValueError("Too few model groups for three partitions.")

    raw = target * number
    sizes = np.floor(raw).astype(int)

    order = np.argsort(
        -(raw - sizes),
        kind="stable"
    )

    sizes[order[:number - sizes.sum()]] += 1

    if (sizes == 0).any():
        raise ValueError("A partition would contain no model groups.")

    return sizes

rng = np.random.default_rng(seed)
best_assignment = None
best_score = float("inf")

for _ in range(trials):
    assignment = np.full(
        len(model_counts), -1, dtype=int
    )

    for body in classes:
        positions = rng.permutation(
            np.flatnonzero(primary_body == body)
        )

        sizes = allocate_models(len(positions))

        groups = np.split(
            positions,
            np.cumsum(sizes)[:-1]
        )

        for split_number, positions_in_split in enumerate(groups):
            assignment[positions_in_split] = split_number

    observed = np.array([
        counts[assignment == split_number].sum(axis=0)
        for split_number in range(3)
    ])

    if (observed == 0).any():
        continue

    score = np.square(
        observed / class_totals - target[:, None]
    ).sum()

    if score < best_score - 1e-12:
        best_assignment = assignment.copy()
        best_score = float(score)

assert best_assignment is not None

model_map = dict(zip(
    model_counts.index,
    split_names[best_assignment]
))

clean["Split"] = clean["Genmodel_ID"].map(model_map)

assert clean["Split"].notna().all()
assert clean.groupby("Genmodel_ID")["Split"].nunique().eq(1).all()
assert clean.groupby("Adv_ID")["Split"].nunique().eq(1).all()
assert clean.groupby("File_SHA256")["Split"].nunique().eq(1).all()

for split in split_names:
    assert set(
        clean.loc[clean["Split"].eq(split), "Body_type"]
    ) == set(classes)

summary = clean.groupby(
    ["Split", "Body_type"]
).agg(
    Listings=("Adv_ID", "nunique"),
    Photos=("Image_ID", "size"),
    Models=("Genmodel_ID", "nunique")
).reindex(
    pd.MultiIndex.from_product(
        [split_names, classes],
        names=["Split", "Body_type"]
    )
)

display(summary)

Listings  Photos  Models
Split      Body_type                          
train      Sedan           275     825      75
           SUV             256     768      88
           Pickup           64     192       6
validation Sedan            63     189      16
           SUV              58     174      19
           Pickup           10      30       2
test       Sedan            57     171      16
           SUV              57     171      19
           Pickup           20      60       1

In [11]:
clean.to_csv(
    output_dir / "manifest_with_splits.csv",
    index=False
)

for split in split_names:
    clean.loc[
        clean["Split"].eq(split)
    ].to_csv(
        output_dir / f"{split}_manifest.csv",
        index=False
    )

assignments = model_counts.reset_index().rename(
    columns={
        body: f"{body}_listings"
        for body in classes
    }
)

assignments["Primary_body_type"] = primary_body
assignments["Split"] = split_names[best_assignment]

assignments.to_csv(
    output_dir / "model_split_assignments.csv",
    index=False
)

summary.to_csv(output_dir / "split_summary.csv")

config = {
    "seed": seed,
    "candidate_allocations": trials,
    "target_proportions": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15
    },
    "grouping_variable": "Genmodel_ID",
    "selection_rule": (
        "Fixed model counts within primary body types; "
        "minimize squared deviation from class-level listing targets"
    ),
    "model_performance_used": False,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__
}

(output_dir / "split_config.json").write_text(
    json.dumps(config, indent=2)
)

totals = clean.groupby("Split").agg(
    Listings=("Adv_ID", "nunique"),
    Photos=("Image_ID", "size"),
    Models=("Genmodel_ID", "nunique")
).reindex(split_names)

assert totals["Photos"].to_dict() == {
    "train": 1785,
    "validation": 393,
    "test": 402
}

display(totals)
print("Passed: complete triplets and no model overlap.")

,Listings,Photos,Models
Split,,,
train,595,1785,168
validation,131,393,37
test,134,402,36


Passed: complete triplets and no model overlap.


In [12]:
output_zip = Path("/content/DVM_Cleaned_Data.zip")

with zipfile.ZipFile(
    output_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(output_dir.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                path.relative_to(output_dir).as_posix()
            )

files.download(str(output_zip))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>